# Imports

In [15]:
import os
import sys
import operator

# To define constants
from typing_extensions import Literal

# To support older python versions
from typing_extensions import TypedDict, Annotated, List, Sequence

# Langgraph Entities to build and design control flow
from langgraph.graph import MessagesState, StateGraph, START, END

# Langchain LLM model integration
from langchain.chat_models import init_chat_model

# For runtime validation on state and output schemas
from pydantic import BaseModel, Field

# Different Message types for defining chat model ip and op
from langchain_core.messages import BaseMessage, HumanMessage

# Chat History Management to append messages
from langgraph.graph.message import add_messages

# Tool oriented imports
from langchain_core.tools import tool, InjectedToolArg
from tavily import AsyncTavilyClient # Web search engine [performs parallel searches][

# Custom Imports
sys.path.append(os.path.abspath("../src"))
from prompts import research_agent_prompt, summarize_webpage_prompt
from utils import format_markdown_messages

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

True

# Defining State and Schemas

## State Definitions

In [2]:
class ResearcherState(TypedDict):
    """
    State for the research agent containing message history and research metadata.
    """
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages]
    tool_call_iterations: int
    research_topic: str
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]

class ResearcherOutputState(TypedDict):
    """
    Output state for the research agent containing final research results.
    """
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages]

## Structured Output Schemas

In [16]:
class ClarifyWithUserSchema(BaseModel):
    """Schema for user clarification decision and questions."""
    
    need_clarification: bool = Field(
        description="Whether the user needs to be asked a clarifying question.",
    )
    question: str = Field(
        description="A question to ask the user to clarify the report scope",
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information.",
    )

class ResearchQuestionSchema(BaseModel):
    """Schema for structured research brief generation."""
    
    research_brief: str = Field(
        description="A research question that will be used to guide the research.",
    )
    
class SummarySchema(BaseModel):
    """Schema for webpage content summarization."""
    summary: str = Field(description="Concise summary of the webpage content")

# Core Research Workflow

*The goal of research is to gather the context requested by the research brief.*

This module implements the core research phase of the research workflow, where we:
1. Conduct web searches using tavily tool
2. Analyze results after every search to check for comprehensiveness
3. Format search results into a well structued summarized output

## Setting Up LLM and Tavily Client

In [19]:
webpage_summarization_llm = init_chat_model(
    "openai:gpt-4.1",
    temperature=0, # Factual response
    max_tokens=1000, # To limit huge responses
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

In [5]:
tavily_client = AsyncTavilyClient()

## Search Utility Functions

In [30]:
async def tavily_search(search_queries, max_results=3, 
                        topic: Literal["general", "news", "finance"] = "general", 
                        include_raw_content=True) -> List[dict]:
    
    """
    Perform search using Tavily API for the given query(ies).

    Args:
        search_queries: List of search queries to execute
        max_results: Maximum number of results per query - defaults to 3
        topic: Topic filter for search results - defaults to general
        include_raw_content: Whether to include raw webpage content

    Returns:
        List of search result dictionaries
    """

    # Asynchronously performs multiple searches parallely
    search_docs = []
    for query in search_queries:
        result = await tavily_client.search(
            query=query,
            max_results=max_results,
            include_raw_content=include_raw_content,
            topic=topic
        )
        search_docs.append(result)

    return search_docs


In [31]:
def summarize_webpage_content(webpage_content: str) -> str:
    """
    Summarize webpage content using the LLM model.
    
    Args:
        webpage_content: Raw webpage content to summarize
        
    Returns:
        Formatted summary with key excerpts
    """
    try:
        # Set up structured output model for summarization
        structured_output_summarizer = webpage_summarization_llm.with_structured_output(Summary)
        
        # Generate summary
        summary = structured_output_summarizer.invoke([HumanMessage(content=summarize_webpage_prompt.format(webpage_content=webpage_content))])
        
        # Format the generated summary with clear structure
        formatted_summary = (
            f"<summary>\n{summary.summary}\n</summary>\n\n"
            f"<key_excerpts>\n{summary.key_excerpts}\n</key_excerpts>"
        )
        
        return formatted_summary
        
    except Exception as e:
        print(f"Failed to summarize webpage: {str(e)}")
        return webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content

In [8]:
def deduplicate_search_results(search_results: List[dict]) -> dict:
    """
    Deduplicate search results by URL to avoid processing duplicate content.
    
    Args:
        search_results: List of search result dictionaries
        
    Returns:
        Dictionary mapping URLs to unique results
    """
    unique_results = {}
    
    for response in search_results:
        for result in response['results']:
            url = result['url']
            if url not in unique_results:
                unique_results[url] = result
    
    return unique_results

In [9]:
def process_search_results(unique_results: dict) -> dict:
    """
    Consolidate summarized content from each of the search results.
    
    Args:
        unique_results: Dictionary of unique search results
        
    Returns:
        Dictionary of processed results with summaries
    """
    summarized_results = {}
    
    for url, result in unique_results.items():
        # Use existing content if no raw content for summarization
        if not result.get("raw_content"):
            content = result['content']
        else:
            # Summarize raw content for better processing
            content = summarize_webpage_content(result['raw_content'])
        
        summarized_results[url] = {
            'title': result['title'],
            'content': content
        }
    
    return summarized_results

In [10]:
def format_search_output(summarized_results: dict) -> str:
    """
    Format summarized search results into a well-structured string output.
    
    Args:
        summarized_results: Dictionary of processed search results
        
    Returns:
        Formatted string of search results with clear source separation
    """
    if not summarized_results:
        return "No valid search results found. Please try different search queries or use a different search API."
    
    formatted_output = "Search results: \n\n"
    
    for i, (url, result) in enumerate(summarized_results.items(), 1):
        formatted_output += f"\n\n--- SOURCE {i}: {result['title']} ---\n"
        formatted_output += f"URL: {url}\n\n"
        formatted_output += f"SUMMARY:\n{result['content']}\n\n"
        formatted_output += "-" * 80 + "\n"
    
    return formatted_output

## Defining Tools

In [33]:
@tool(parse_docstring=True)
def tavily_search_tool(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 3,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general") -> str:
    """
    Fetch results from Tavily search API and apply content summarization logic.
    InjectedToolArg: To ensure that these pasrams are hidden to the LLM to avoid hallucinations

    Args:
        query: A single search query to execute
        max_results: Maximum number of results to return
        topic: Topic to filter results by ('general', 'news', 'finance')

    Returns:
        Formatted string of search results with summaries
    """
    # Execute search for the query
    search_results = tavily_search([query],
                                    max_results=max_results,
                                    topic=topic,
                                    include_raw_content=True)

    # Deduplicate results by URL to avoid processing duplicate content
    unique_results = deduplicate_search_results(search_results)

    # Prepare summarization of each of the search results
    summarized_results = process_search_results(unique_results)

    # Format output for consumption
    return format_search_output(summarized_results)

In [14]:
@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.
    
    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.
    
    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?
    
    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps
        
    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

## Defining Workflow

### Configuration

In [27]:
# LLM responsible for clening the consolidated summary
master_summarization_llm = init_chat_model(
    "openai:gpt-4.1",
    temperature=0, # Factual response
    max_tokens=32000, # Set to larger limit to accomodate summaries from multiple sources
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

In [28]:
# LLM to handle the core research
research_agent_llm = init_chat_model(
    "anthropic:claude-sonnet-4-20250514",
    temperature=0, # Factual response
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [35]:
# Set up tools and model binding
tools = [tavily_search_tool, think_tool]
tools_by_name = {tool.name: tool for tool in tools}

research_agent_llm_with_tools = research_agent_llm.bind_tools(tools)

### Defining Nodes

In [ ]:
def call_research_agent_llm(state: ResearcherState):
    """Analyze current state and decide on next actions.
    
    The model analyzes the current conversation state and decides whether to:
    1. Call search tools to gather more information
    2. Provide a final answer based on gathered information
    
    Returns updated state with the model's response.
    """
    return {
        "researcher_messages": [
            model_with_tools.invoke(
                [SystemMessage(content=research_agent_prompt)] + state["researcher_messages"]
            )
        ]
    }